Date: 01/02/2024 <br>
Desc: To test BN model for UAV communication reliability prediction, using pandas crosstab to populate CPTs.

In [11]:
import pandas as pd
import numpy as np 
import math
import os
from tqdm import tqdm
from sklearn.metrics import accuracy_score

def get_mcs_index(df_in):
    '''
    Gets the MCS index based on modulation and bitrate column of the df_in
    '''
    df = df_in.copy()
    df["MCS"] = ''
    df.loc[(df["Modulation"] == "BPSK") & (df["Bitrate"] == 6.5), "MCS"] = 0 # MCS Index 0
    df.loc[(df["Modulation"] == "QPSK") & (df["Bitrate"] == 13), "MCS"] = 1 # MCS Index 0
    df.loc[(df["Modulation"] == "QPSK") & (df["Bitrate"] == 19.5), "MCS"] = 2 # MCS Index 0
    df.loc[(df["Modulation"] == "QAM16") & (df["Bitrate"] == 26), "MCS"] = 3 # MCS Index 0
    df.loc[(df["Modulation"] == "QAM16") & (df["Bitrate"] == 39), "MCS"] = 4 # MCS Index 0
    df.loc[(df["Modulation"] == "QAM64") & (df["Bitrate"] == 52), "MCS"] = 5 # MCS Index 0
    df.loc[(df["Modulation"] == "QAM64") & (df["Bitrate"] == 58.5), "MCS"] = 6 # MCS Index 0
    df.loc[(df["Modulation"] == "QAM64") & (df["Bitrate"] == 65), "MCS"] = 7 # MCS Index 0

    return df

def find_nearest_value(value, array):
    array = np.asarray(array)
    idx = (np.abs(array - value)).argmin()
    return array[idx]

def find_nearest_index(value, array):
    array = np.asarray(array)
    idx = (np.abs(array - value)).argmin()
    return idx


In [12]:
# Set paths to datasets etc.
CPT_PATH = "/media/research-student/KingstonSSD/FANET_Dataset/Dataset_NP100000_DJISpark/bn_cpts/djispark_reliability_bn_CPT_Downlink.csv"
HDIST_BIN_PATH = "/media/research-student/One Touch/FANET_Dataset/Dataset_NP10000_DJISpark/bn_ckpt/djispark_reliability_bn_hdist_bins_Downlink.npy"
HEIGHT_BIN = "/media/research-student/One Touch/FANET_Dataset/Dataset_NP10000_DJISpark/bn_ckpt/djispark_reliability_bn_height_bins_Downlink.npy"
TEST_DATA_PATH = "/media/research-student/KingstonSSD/FANET_Dataset/Dataset_NP100000_DJISpark/test_dataset_1_processed/Downlink_Reliability.csv"
# Set minimum probability for failure mode to be considered
MIN_FAILURE_PROB = 0.01
RELIABILITY_TH = [0.95, 0.99, 0.999] # For calculation max AE in reliable region

# Load CPT and bin files
cpt_df = pd.read_csv(CPT_PATH)
# hdist_bin = np.load(HDIST_BIN_PATH)
# height_bin = np.load(HEIGHT_BIN)
hdist_bin = np.arange(0, 710, 10) # For associating each hdist to its nearest value in train dataset
height_bin = np.arange(60, 330, 30) # For associating each height to its nearest value in train dataset

# Load testing dataset and process
test_data_df = pd.read_csv(TEST_DATA_PATH)
test_data_df = get_mcs_index(test_data_df)
    # If associating each horizontal distance and height with the closest values in training dataset:
test_data_df["Horizontal_Distance_Class"] = test_data_df["Horizontal_Distance"].apply(find_nearest_index, args=([hdist_bin]))
test_data_df["Height_Class"] = test_data_df["Height"].apply(find_nearest_index, args=([height_bin]))
test_data_df["UAV_Sending_Interval_Class"] = test_data_df["UAV_Sending_Interval"].replace({10:0, 20:1, 66.7:2, 100:3}) # Change sending interval categorial to numeric
test_data_df["Reliability"] = (test_data_df["Num_Reliable"] / test_data_df["Num_Sent"]).values
test_data_df["Delay_Excd_Prob"] = (test_data_df["Num_Delay_Excd"] / test_data_df["Num_Sent"]).values
test_data_df["Queue_Overflow_Prob"] = (test_data_df["Num_Q_Overflow"] / test_data_df["Num_Sent"]).values
test_data_df["Incr_Rcvd_Prob"] = (test_data_df["Num_Incr_Rcvd"] / test_data_df["Num_Sent"]).values

predicted_reliability = [] # To store reliability predictions
predicted_incr_rcvd = [] # To store incorrectly received probability predictions
predicted_delay_excd = [] # To store delay exceeded probability predictions
predicted_q_ovflw = [] # To store queue overflow probability predictions
for row in tqdm(test_data_df.itertuples()):
    # predictions = cpt_df.loc[(row.Horizontal_Distance_Class,row.Height_Class,row.UAV_Sending_Interval_Class,row.MCS)]
    predictions = cpt_df.loc[(cpt_df["Horizontal_Distance_Class"]==row.Horizontal_Distance_Class) & (cpt_df["Height_Class"]==row.Height_Class) &
                             (cpt_df["UAV_Sending_Interval_Class"]==row.UAV_Sending_Interval_Class) & (cpt_df["MCS"]==row.MCS)]
    try:
        predicted_reliability.append(predictions["0"].values[0])
        predicted_delay_excd.append(predictions["1"].values[0])
        predicted_q_ovflw.append(predictions["2"].values[0])
        predicted_incr_rcvd.append(predictions["3"].values[0])
    except:
        print(row)
        print(predictions)
        break

test_data_df['Predicted_Reliability'] = predicted_reliability
test_data_df['Predicted_Delay_Excd_Prob'] = predicted_delay_excd
test_data_df['Predicted_Queue_Overflow_Prob'] = predicted_q_ovflw
test_data_df['Predicted_Incr_Rcvd_Prob'] = predicted_incr_rcvd

test_data_df["Reliability_Class"] = pd.cut(test_data_df["Reliability"], bins=[-0.1,0.1,0.5,0.9,1], labels=["Low", "ModeratelyLow", "ModeratelyHigh", "High"])
test_data_df["Predicted_Reliability_Class"] = pd.cut(test_data_df["Predicted_Reliability"], bins=[-0.1,0.1,0.5,0.9,1], labels=["Low", "ModeratelyLow", "ModeratelyHigh", "High"])
test_data_df["Failure_Mode"] = test_data_df[["Queue_Overflow_Prob", "Incr_Rcvd_Prob", "Delay_Excd_Prob"]].idxmax(axis=1)
test_data_df["Predicted_Failure_Mode"] = test_data_df[["Predicted_Queue_Overflow_Prob", "Predicted_Incr_Rcvd_Prob", "Predicted_Delay_Excd_Prob"]].idxmax(axis=1)
# Replace label for Failure Mode with "None" if none of the failure modes have a probability > 5%
test_data_df.loc[(test_data_df["Queue_Overflow_Prob"] < MIN_FAILURE_PROB) & (test_data_df["Incr_Rcvd_Prob"] < MIN_FAILURE_PROB) & (test_data_df["Delay_Excd_Prob"] < MIN_FAILURE_PROB),["Failure_Mode"]] = "None"
test_data_df.loc[(test_data_df["Predicted_Queue_Overflow_Prob"] < MIN_FAILURE_PROB) & (test_data_df["Predicted_Incr_Rcvd_Prob"] < MIN_FAILURE_PROB) & (test_data_df["Predicted_Delay_Excd_Prob"] < MIN_FAILURE_PROB),["Predicted_Failure_Mode"]] = "None"

# Compute the model accuracy and mean abs err
failure_mode = test_data_df["Failure_Mode"].replace({"Queue_Overflow_Prob":1, "Incr_Rcvd_Prob":2, "Delay_Excd_Prob":3, "None":4})
failure_mode_predicted = test_data_df["Predicted_Failure_Mode"].replace({"Predicted_Queue_Overflow_Prob":1, "Predicted_Incr_Rcvd_Prob":2, "Predicted_Delay_Excd_Prob":3, "None":4})
reliability_accuracy = accuracy_score(test_data_df["Reliability_Class"], test_data_df["Predicted_Reliability_Class"])
failure_mode_accuracy = accuracy_score(failure_mode, failure_mode_predicted)
reliability_mae = np.mean(abs(test_data_df['Reliability'].values - test_data_df['Predicted_Reliability'].values))
queue_overflow_mae = np.mean(abs(test_data_df['Queue_Overflow_Prob'].values - test_data_df['Predicted_Queue_Overflow_Prob'].values))
incr_rcvd_mae = np.mean(abs(test_data_df['Incr_Rcvd_Prob'].values - test_data_df['Predicted_Incr_Rcvd_Prob'].values))
delay_excd_mae = np.mean(abs(test_data_df['Delay_Excd_Prob'].values - test_data_df['Predicted_Delay_Excd_Prob'].values))
reliability_maxae = np.max(abs(test_data_df['Reliability'].values - test_data_df['Predicted_Reliability'].values))
queue_overflow_maxae = np.max(abs(test_data_df['Queue_Overflow_Prob'].values - test_data_df['Predicted_Queue_Overflow_Prob'].values))
incr_rcvd_maxae = np.max(abs(test_data_df['Incr_Rcvd_Prob'].values - test_data_df['Predicted_Incr_Rcvd_Prob'].values))
delay_excd_maxae = np.max(abs(test_data_df['Delay_Excd_Prob'].values - test_data_df['Predicted_Delay_Excd_Prob'].values))

# Print results
print("Reliability - Accuracy: {}, MAE: {}, MaxAE: {}".format(reliability_accuracy, reliability_mae, reliability_maxae))
print("Failure Mode - Accuracy: {}".format(failure_mode_accuracy))
print("Queue Overflow - MeanAE: {}, MaxAE: {}".format(queue_overflow_mae, queue_overflow_maxae))
print("Incorrectly Received - MeanAE: {}, MaxAE: {}".format(incr_rcvd_mae, incr_rcvd_maxae))
print("Delay Exceeded - MeanAE: {}, MaxAE: {}".format(delay_excd_mae, delay_excd_maxae))
print("Average Failure Mode Mean AE: {}".format(np.mean([queue_overflow_mae, incr_rcvd_mae, delay_excd_mae])))

for reliability_th in RELIABILITY_TH:
    # Get the Max Abs Err of reliability, but only when either the simulated/predicted reliability is above the threshold
    # test_data_reliable_df = test_data_df.loc[(test_data_df["Reliability"]>=reliability_th) | (test_data_df["Predicted_Reliability"]>=reliability_th)]
    # rel_err = test_data_reliable_df['Reliability'].values - test_data_reliable_df['Predicted_Reliability'].values
    # reliability_maxae_reliable = np.max(abs(rel_err))
    # reliability_mean_reliable = np.mean(abs(rel_err))
    # print("Reliability - MeanAE_Reliable_Region: {}, MaxAE_Reliable_Region: {}".format(reliability_mean_reliable, reliability_maxae_reliable))

    # UNCOMMENT TO EVALUATE reliability_state_accuracy over reliable/predicted_reliable region only
    # Get the accuracy of predicting reliability above the threshold
    # test_data_df["Reliable_State"] = test_data_df["Reliability"] >= reliability_th
    # test_data_df["Predicted_Reliable_State"] = test_data_df["Predicted_Reliability"] >= reliability_th
    # reliability_state_accuracy = accuracy_score(test_data_df["Reliable_State"], test_data_df["Predicted_Reliable_State"])

    # # UNCOMMENT TO EVALUATE reliability_state_accuracy over entire region
    # # Get the accuracy of predicting reliability above the threshold
    test_data_df["Reliable_State"] = test_data_df["Reliability"] >= reliability_th
    test_data_df["Predicted_Reliable_State"] = test_data_df["Predicted_Reliability"] >= reliability_th
    reliability_state_accuracy = accuracy_score(test_data_df["Reliable_State"], test_data_df["Predicted_Reliable_State"])
    print("Reliability - Accuracy_Reliable_Region >= {}: {}".format(reliability_th, reliability_state_accuracy))

# Save results to file
# test_data_df.to_csv("/media/research-student/One Touch/FANET_Dataset/Dataset_NP10000_DJIMavicAir/Test_Dataset_2_NP10000_DJIMavicAir_Downlink_Reliability_RESULTS_bn_pandas_cpt.csv")

0it [00:00, ?it/s]

Pandas(Index=0, Horizontal_Distance=5.0, Height=75, UAV_Sending_Interval=10.0, Modulation='BPSK', Bitrate=6.5, Mean_SINR=701.2762069874864, Std_Dev_SINR=293.6643258541574, Num_Sent=100000, Num_Reliable=0, Num_Delay_Excd=25192, Num_Incr_Rcvd=1, Num_Q_Overflow=74807, MCS=0, Horizontal_Distance_Class=0, Height_Class=0, UAV_Sending_Interval_Class=0.0, Reliability=0.0, Delay_Excd_Prob=0.25192, Queue_Overflow_Prob=0.74807, Incr_Rcvd_Prob=1e-05)
   Horizontal_Distance_Class  Height_Class  UAV_Sending_Interval_Class  MCS  \
0                          0             0                           0    0   

   Reliability  Prob_Delay_Excd  Prob_Queue_Overflow  Num_Incr_Rcvd  
0      0.00009          0.25013              0.74976        0.00002  


ValueError: Length of values (0) does not match length of index (3840)

In [12]:
test_data_reliable_df

,Horizontal_Distance,Height,UAV_Sending_Interval,Modulation,Bitrate,Mean_SINR,Std_Dev_SINR,Num_Sent,Num_Reliable,Num_Delay_Excd,...,Predicted_Reliability,Predicted_Delay_Excd_Prob,Predicted_Queue_Overflow_Prob,Predicted_Incr_Rcvd_Prob,Reliability_Class,Predicted_Reliability_Class,Failure_Mode,Predicted_Failure_Mode,Reliable_State,Predicted_Reliable_State
600,5,285,20.0,QAM16,26.0,49.422382,20.532534,10000,10000,0,...,1.0000,0.0000,0.0,0.0,High,High,None,None,True,True
601,15,285,20.0,QAM16,26.0,48.662958,20.324580,10000,10000,0,...,1.0000,0.0000,0.0,0.0,High,High,None,None,True,True
602,25,285,20.0,QAM16,26.0,47.800266,20.071453,10000,10000,0,...,1.0000,0.0000,0.0,0.0,High,High,None,None,True,True
603,35,285,20.0,QAM16,26.0,46.841569,19.775562,10000,10000,0,...,1.0000,0.0000,0.0,0.0,High,High,None,None,True,True
604,45,285,20.0,QAM16,26.0,45.795377,19.439886,10000,10000,0,...,1.0000,0.0000,0.0,0.0,High,High,None,None,True,True
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1838,385,225,100.0,QPSK,13.0,11.032399,6.530227,10000,10000,0,...,0.9989,0.0011,0.0,0.0,High,High,None,None,True,True
1839,395,225,100.0,QPSK,13.0,10.452224,6.258595,10000,9995,5,...,0.9986,0.0014,0.0,0.0,High,High,None,None,True,True
1840,405,225,100.0,QPSK,13.0,9.908005,6.001860,10000,9990,10,...,0.9974,0.0026,0.0,0.0,High,High,None,None,True,True
1841,415,225,100.0,QPSK,13.0,9.397294,5.759086,10000,9966,34,...,0.9968,0.0032,0.0,0.0,High,High,None,None,True,True
